# Qubit-Medic - end-to-end Colab notebook

Runs SFT warm-up + GRPO RL on a single Colab T4. Total wall-clock: ~24 hours
(SFT ~30 min, GRPO ~22 hours, eval ~30 min). The notebook is structured so
every cell is idempotent and re-runnable.

**W&B integration is on by default.** Every stage (format-test, SFT, GRPO,
eval) logs to the same W&B project (`qubit-medic`) and shares a `--wandb-group`
so the runs appear together in the dashboard. Set `WANDB_DISABLED=1` if you
want to skip W&B entirely.

## 1. Clone the repo and install

In [ ]:
%cd /content
!git clone https://github.com/qubit-medic/qubit-medic.git || (cd qubit-medic && git pull)
%cd qubit-medic
!pip install -q -r requirements.txt
!pip install -q -r requirements-train.txt
!pip install -q --no-deps unsloth

## 2. Configure W&B

Paste your API key from <https://wandb.ai/authorize>. The `EXPERIMENT_GROUP`
below is what bundles the format-test, SFT, GRPO, and eval runs together
on the dashboard - bump it for each new experiment.

In [ ]:
import os, datetime
EXPERIMENT_GROUP = f"colab-{datetime.datetime.utcnow().strftime('%Y%m%d-%H%M')}"
os.environ['WANDB_PROJECT'] = 'qubit-medic'
# os.environ['WANDB_ENTITY'] = 'your-team'        # uncomment if you use a team
# os.environ['WANDB_DISABLED'] = '1'              # uncomment to skip W&B
print('experiment group:', EXPERIMENT_GROUP)
!wandb login

## 3. Validate the environment

All five gates must pass before going further. (No W&B logging here - this
is a static check.)

In [ ]:
!python -m scripts.validate_env

## 4. Section 1.3 - format-test (existential go/no-go)

If parseable rate is below 30%, SFT is mandatory. The result is logged to
W&B under `format_test/*`.

In [ ]:
!python -m scripts.format_test \
    --backend unsloth \
    --model Qwen/Qwen2.5-3B-Instruct \
    --syndromes 10 --samples-per 3 \
    --out data/format_test.json \
    --report-to wandb \
    --wandb-group {EXPERIMENT_GROUP}

## 5. Generate SFT data (5,000 syndromes, ~5 min)

In [ ]:
!python -m scripts.generate_sft_data --n 5000 --out data/sft_dataset.jsonl

## 6. SFT warm-up (~30 min on T4)

Logs `sft/loss`, `sft/parse_success_rate`, and a `sft/generations` table
every 100 steps. Uploads the LoRA adapter dir as a W&B artifact at the end.

In [ ]:
!python -m scripts.train_sft \
    --dataset data/sft_dataset.jsonl \
    --output checkpoints/sft_warmup \
    --report-to wandb \
    --wandb-group {EXPERIMENT_GROUP} \
    --wandb-run-name sft-warmup-{EXPERIMENT_GROUP} \
    --wandb-notes 'SFT warm-up on PyMatching-derived syndromes' \
    --sample-every 100 --sample-count 4

## 7. SFT validation gate (Section 6.2)

In [ ]:
!python -m scripts.eval \
    --adapter checkpoints/sft_warmup \
    --episodes 100 \
    --out data/sft_eval.json \
    --report-to wandb \
    --wandb-group {EXPERIMENT_GROUP} \
    --wandb-run-name eval-sft-{EXPERIMENT_GROUP}

## 8. GRPO RL training (~22 hours on T4)

Logs `rl/reward/<component>_mean|std|min|max` for each of the five reward
components, `rl/parse/*`, `rl/curriculum/*`, plus a generation table and
an in-loop greedy eval every 200 steps. Uploads the trained adapter as a
W&B artifact at the end.

Adjust `--steps` if your time budget is tighter (~250 steps/hour on a T4).

In [ ]:
!python -m scripts.train_grpo \
    --sft-checkpoint checkpoints/sft_warmup \
    --output checkpoints/grpo \
    --steps 2000 \
    --report-to wandb \
    --wandb-group {EXPERIMENT_GROUP} \
    --wandb-run-name grpo-{EXPERIMENT_GROUP} \
    --wandb-notes 'GRPO with 5 verifiable rewards' \
    --sample-every 50 --sample-n 8 \
    --inloop-eval-every 200 --inloop-eval-episodes 50

## 9. Final evaluation + headline plots

In [ ]:
!python -m scripts.eval \
    --adapter checkpoints/grpo --episodes 500 \
    --out data/grpo_eval.json \
    --report-to wandb \
    --wandb-group {EXPERIMENT_GROUP} \
    --wandb-run-name eval-grpo-{EXPERIMENT_GROUP}

!python -m scripts.baseline_policies --episodes 500 --out data/baseline_results.json
!python -m scripts.plot_results --baselines data/baseline_results.json --out-dir figures
!python -m scripts.animate_grid --frames 50

## 10. Optional: Willow real-chip cross-validation (Section 8)

In [ ]:
# Manually download from https://zenodo.org/record/13359217 and place at data/willow_d3.dem
!python -m scripts.willow_validation --dem data/willow_d3.dem --episodes 1000

## 11. Push to Hugging Face Spaces

After successful training, push the env + adapters to a Space.

In [ ]:
from huggingface_hub import HfApi, login
login()  # paste your HF token
api = HfApi()
# Replace with your Space repo id.
api.upload_folder(folder_path='.', repo_id='your-team/qubit-medic', repo_type='space')